# Kev-0.8B: чистый старт и остановка по валидации

Продолжаем обучение закреплённых исходных LoRA и pointer head. Базовая Qwen заморожена. Те же 22 656 строк и E5 top-20: первые 2 272 clean, затем остаток clean/STT; максимум один проход. Это один запуск адаптации, без сравнения двух curriculum. Личный каталог и test остаются локально.

## Среда

Native Kev code закреплён в архиве; исходные base, adapter и head проверяются по SHA-256. LoRA не сливается с базой. Training, weights и validation FP32: на T4 FP16 backward переполнился до первого обновления; его результаты не используются.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Kaggle ships torchao 0.10; PEFT 0.19 rejects it even for plain FP32 LoRA.
# This experiment uses no torchao quantization. Remove only that optional package.
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'peft==0.19.0', 'transformers==5.6.2', 'safetensors==0.6.2', 'accelerate==1.15.0', 'pydantic==2.12.5'])
import torch
assert torch.cuda.is_available(), 'Нужен GPU'
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0)})


## Обучение

LoRA и pointer head: AdamW lr=2e-5, effective batch 16 (microbatch 1 × accumulation 16), warmup 6%, gradient clipping 1.0. Полный запрос и 20 кандидатов сохраняются; overflow останавливает запуск. Проверка каждые 142 updates: group-balanced conditional NLL с равным весом basic/difficult/ASR. min_delta=0.005, patience=3; baseline может остаться лучшей.

In [ ]:
matches = list(Path('/kaggle/input').rglob('kev-input-manifest.json'))
assert len(matches) == 1
input_dir = matches[0].parent
import hashlib
assert hashlib.sha256((input_dir / 'kev-input-manifest.json').read_bytes()).hexdigest() == 'b9dafdb3b2cd861c7c8b82a35db8d98cec56c9f7bc0438d5cff40aef7fb5a057', 'Unexpected input dataset version'
output_dir = Path('/kaggle/working/results')
command = [sys.executable, str(input_dir / 'train_kev_kaggle.py'), '--input', str(input_dir), '--output', str(output_dir)]
with Path('/kaggle/working/training.log').open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    assert process.wait() == 0, 'См. training.log'


## Выбранные веса

checkpoint — выбранные LoRA/head; last-checkpoint — последние. После скачивания нужна парная CPU-проверка. Сохранён temperature=1.0: confidence не откалиброван. Прочие typed-задачи Kev, GUI и CPU latency этим запуском не проверены.

In [ ]:
result = json.loads((output_dir / 'result.json').read_text())
print({key: result[key] for key in ('steps', 'best_step', 'stop_reason', 'elapsed_seconds')})
for profile in result['baseline']['metrics']:
    before = result['baseline']['metrics'][profile]['accuracy']
    after = result['best']['metrics'][profile]['accuracy']
    print(f'{profile}: {before:.1%} -> {after:.1%}')
